In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [5]:
class PneumoniaDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        self.labels = []
        self.transform = transform

        for label_dir in ['NORMAL', 'PNEUMONIA']:
            class_path = os.path.join(root_dir, label_dir)
            if not os.path.exists(class_path):
                continue

            for img_name in os.listdir(class_path):
                if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue

                self.image_paths.append(os.path.join(class_path, img_name))

                if label_dir == 'NORMAL':
                    self.labels.append(0)
                elif 'bacteria' in img_name.lower():
                    self.labels.append(1)
                else:
                    self.labels.append(2)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, self.labels[idx]

In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_ds = PneumoniaDataset("train", transform=transform)
test_ds = PneumoniaDataset("test", transform=transform)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=4)

print("Train:", len(train_ds), "Test:", len(test_ds))

Train: 1082 Test: 234


In [7]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 3)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Rutika/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:25<00:00, 829kB/s] 


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 5   # ✅ REDUCED

In [9]:
print("Training started...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} Loss: {running_loss:.4f}")

Training started...
Epoch 1/5 Loss: 25.8877
Epoch 2/5 Loss: 0.4793
Epoch 3/5 Loss: 0.1970
Epoch 4/5 Loss: 0.1150
Epoch 5/5 Loss: 0.0712


In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 100.00%


In [11]:
torch.save(model.state_dict(), "pneumonia_model.pth")
print("Model saved!")

Model saved!


In [12]:
def predict(img_path):
    model.eval()
    
    img = Image.open(img_path).convert("RGB")
    img = transform(img)
    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        _, pred = torch.max(outputs, 1)

    classes = ["NORMAL", "BACTERIAL", "VIRAL"]
    print("Prediction:", classes[pred.item()])

predict("test.jpg.jpeg")

Prediction: NORMAL
